# 02. Standardized Evaluation Pipeline (Colab A100)

**Purpose**: Reproducible, standardized evaluation across all EXAONE civil complaint experiments.

**Key Design Decisions**:
1. **transformers 4.44~4.49**: LoRA 학습 시점과 동일한 버전 사용
2. **EXAONE revision 고정** (`17b70148e344`): 2026-02 코드 재작성 이전의 학습 호환 코드 사용
3. **패치 불필요**: 학습 시점 버전 조합이면 monkey-patch 없이 동작
4. **베이스 모델 + LoRA**: `EXAONE-Deep-7.8B` + `civil-complaint-exaone-lora`
5. **SacreBLEU 경량 정규화**: 문장부호를 유지하여 표준 BLEU 점수 보장

**Supported modes**:
- Mode A: HuggingFace (QLoRA adapter)
- Mode B: vLLM (AWQ quantized model)

**Metrics**: SacreBLEU, ROUGE-L (F1), BERTScore (F1), Latency, Throughput

---

## 1. Setup & Configuration

In [ ]:
# === 1-A. 패키지 설치 ===
# LoRA 학습 시점(2025-03)에는 transformers 4.44~4.49 + EXAONE 초기 코드 사용
# → 최신 transformers는 EXAONE 최신 modeling 코드와만 호환되므로 4.44로 고정

!pip uninstall -y transformers accelerate -q
!pip install -q "transformers>=4.44,<4.50" "accelerate>=1.3.0,<2.0" peft
!pip install -q "numpy>=1.26,<3" "pandas>=2.2.2" sacrebleu rouge_score bert_score wandb tqdm bitsandbytes tabulate "protobuf<5.0.0"

# 설치 확인
import transformers, peft
v = tuple(int(x) for x in transformers.__version__.split('.')[:2])
assert v >= (4, 44) and v < (4, 50), \
    f"transformers >=4.44,<4.50 필요! 현재: {transformers.__version__}"
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")

print("\n설치 완료! [런타임] -> [세션 다시 시작] 후 아래 셀부터 진행하세요.")

In [ ]:
# === 1-B. 환경 초기화 ===
# transformers 4.44~4.49 + EXAONE 초기 revision → 패치 최소화
import os, sys, torch, transformers, wandb

print(f'transformers {transformers.__version__}')
print(f'torch {torch.__version__}')

# EXAONE 모델의 학습 시점 revision 고정
# 2026-02-06 커밋에서 modeling_exaone.py가 transformers v5용으로 재작성됨
# 학습 시 사용된 코드는 2025-03-18 Initial commit (e3f42b18f6b1)
EXAONE_REVISION = "17b70148e344"  # 2025-03-19: 학습 호환 마지막 revision

# ──────────────────────────────────────────────
if not os.path.exists('/content/GovOn'):
    !git clone https://github.com/GovOn-Org/GovOn.git /content/GovOn
os.chdir('/content/GovOn')
PROJECT_ROOT = '/content/GovOn'
TEST_PATH = f'{PROJECT_ROOT}/data/processed/civil_complaint_test_with_thought.jsonl'

if torch.cuda.is_available():
    print(f'\nGPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

WANDB_API_KEY = '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'
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY, relogin=True)
print('W&B logged in.\n환경 초기화 완료.')

In [ ]:
# ============================================================
# 1-C. Configuration
# ============================================================
import torch
from dataclasses import dataclass, field
from typing import Optional, List, Dict


@dataclass
class EvalConfig:
    """Central configuration for the evaluation pipeline."""

    # --- Inference mode ---
    inference_mode: str = "hf"

    # --- Model paths ---
    base_model_id: str = "LGAI-EXAONE/EXAONE-Deep-7.8B"
    lora_adapter_id: str = "umyunsang/civil-complaint-exaone-lora"
    awq_model_id: str = "umyunsang/civil-complaint-exaone-awq"

    # --- Data ---
    test_data_path: str = ""

    # --- Generation parameters ---
    max_new_tokens: int = 256  # 참조 답변 평균 길이 기반 (512 → 256 축소)
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.1
    do_sample: bool = False

    # --- System message (학습 노트북과 동일) ---
    system_message: str = "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."

    # --- BERTScore model ---
    bertscore_model: str = "bert-base-multilingual-cased"

    # --- vLLM settings ---
    vllm_max_model_len: int = 4096
    vllm_gpu_memory_utilization: float = 0.80

    # --- W&B ---
    wandb_project: str = "govon-evaluation"
    wandb_entity: Optional[str] = None
    wandb_run_name: Optional[str] = None
    wandb_tags: List[str] = field(default_factory=lambda: ["evaluation"])

    # --- Grid search ---
    grid_search_enabled: bool = False
    grid_search_n_samples: int = 50

    # --- Output ---
    output_dir: str = ""

    def __post_init__(self):
        if not self.test_data_path:
            self.test_data_path = f"{PROJECT_ROOT}/data/processed/civil_complaint_test_with_thought.jsonl"
        if not self.output_dir:
            self.output_dir = f"{PROJECT_ROOT}/docs/outputs/experiments"
        os.makedirs(self.output_dir, exist_ok=True)


config = EvalConfig()

print(f"Inference mode  : {config.inference_mode}")
print(f"Base model      : {config.base_model_id}")
print(f"LoRA adapter    : {config.lora_adapter_id}")
print(f"Max new tokens  : {config.max_new_tokens}")
print(f"Test data       : {config.test_data_path}")
print(f"System message  : {config.system_message}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Test Data Loading

## 3. Text Preprocessing

In [ ]:
import re


# PII token patterns that appear in the dataset
_PII_PATTERNS = [
    r'▲+',
    r'\[NAME_MASKED\]',
    r'\[이름\]',
    r'\[전화번호\]',
    r'\[주소\]',
    r'<NAME>',
    r'<MOBILE_NUMBER>',
    r'<PHONE_NUMBER>',
    r'<DATE>',
    r'<BIRTH_NUMBER>',
    r'<ACCOUNT_NUMBER>',
    r'<ADDRESS>',
]
_PII_REGEX = re.compile('|'.join(_PII_PATTERNS))

# Thought tag pattern
_THOUGHT_REGEX = re.compile(r'<thought>.*?</thought>', flags=re.DOTALL)


def remove_thought_tags(text: str) -> str:
    """Remove <thought>...</thought> blocks from text."""
    return _THOUGHT_REGEX.sub('', text)


def remove_pii_tokens(text: str) -> str:
    """Remove PII placeholder tokens and triangle symbols."""
    return _PII_REGEX.sub('', text)


def normalize_light(text: str) -> str:
    """경량 정규화: SacreBLEU용 (문장부호 유지).
    SacreBLEU는 자체 토크나이저가 있으므로 문장부호를 제거하면 안 됩니다.
    1. Remove <thought> tags
    2. Remove PII tokens
    3. Collapse whitespace
    """
    text = remove_thought_tags(text)
    text = remove_pii_tokens(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def normalize_text(text: str) -> str:
    """Full normalization for ROUGE-L / BERTScore.
    1. Remove <thought> tags
    2. Remove PII tokens
    3. Remove punctuation
    4. Collapse whitespace
    """
    text = remove_thought_tags(text)
    text = remove_pii_tokens(text)
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def korean_tokenize(text: str) -> List[str]:
    """Space-based Korean tokenization."""
    try:
        import mecab
        m = mecab.MeCab()
        return [tok for tok in m.morphs(text) if tok.strip()]
    except (ImportError, Exception):
        return text.split()


# --- Verify preprocessing ---
sample_raw = """<thought>
1. Complaint Type Analysis: blah blah
</thought>
안녕하십니까. [NAME_MASKED] 님, <NAME> 담당자입니다. <DATE>에 접수된 건입니다."""

print(f"Raw:              {sample_raw[:80]}...")
print(f"Light (for BLEU): {normalize_light(sample_raw)}")
print(f"Full  (for ROUGE): {normalize_text(sample_raw)}")

In [ ]:
import json
import pandas as pd
from collections import Counter


def load_test_data(path: str) -> List[Dict]:
    """Load JSONL test set. Each line must have: id, instruction, input, output, category.
    Fields: id, instruction, input, output, category, source."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"WARNING: Skipping line {line_no} - JSON decode error: {e}")
                continue

            # Use category field directly if available; fallback to parsing input
            if "category" not in item or not item["category"]:
                import re
                cat_match = re.search(r'\[카테고리:\s*([^\]]+)\]', item.get("input", ""))
                if not cat_match:
                    cat_match = re.search(r'\[Category:\s*([^\]]+)\]', item.get("input", ""))
                item["category"] = cat_match.group(1).strip() if cat_match else "unknown"

            data.append(item)
    return data


test_data = load_test_data(config.test_data_path)
print(f"Loaded {len(test_data)} test samples from {config.test_data_path}")

# Statistics
cat_counts = Counter(item["category"] for item in test_data)
print(f"\nCategory distribution:")
for cat, count in cat_counts.most_common():
    print(f"  {cat:20s} : {count:5d} ({count / len(test_data) * 100:.1f}%)")

# Reference length stats
ref_lengths = [len(item["output"]) for item in test_data]
print(f"\nReference output length (chars):")
print(f"  mean={sum(ref_lengths)/len(ref_lengths):.0f}, "
      f"min={min(ref_lengths)}, max={max(ref_lengths)}, "
      f"median={sorted(ref_lengths)[len(ref_lengths)//2]}")

# Show first sample
print(f"\n--- Sample 0 ---")
print(f"  ID       : {test_data[0]['id']}")
print(f"  Category : {test_data[0]['category']}")
print(f"  Instruct : {test_data[0]['instruction'][:80]}...")
print(f"  Input    : {test_data[0]['input'][:120]}...")
print(f"  Output   : {test_data[0]['output'][:120]}...")

## 4. Model Loading

In [ ]:
def load_model_hf(cfg: EvalConfig):
    """Base 모델 + LoRA 어댑터 방식으로 로드
    학습 시점의 EXAONE revision을 사용하여 modeling 코드 호환성 보장
    """
    global model, tokenizer
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    print(f"  Base model: {cfg.base_model_id}")
    print(f"  Revision: {EXAONE_REVISION}")
    print(f"  LoRA adapter: {cfg.lora_adapter_id}")

    # 1. 토크나이저 (학습 시점 revision)
    tokenizer = AutoTokenizer.from_pretrained(
        cfg.base_model_id, trust_remote_code=True, revision=EXAONE_REVISION
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # 2. Base 모델 (4-bit 양자화, 학습 시점 revision)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_id,
        revision=EXAONE_REVISION,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )

    # 3. LoRA 어댑터 적용
    model = PeftModel.from_pretrained(base_model, cfg.lora_adapter_id)
    model.eval()

    print(f"  eos_token_id: {tokenizer.eos_token_id}")
    print(f"  pad_token_id: {tokenizer.pad_token_id}")
    print(f"  vocab_size: {len(tokenizer)}")
    print("HF Model (base + LoRA) loaded.")


def load_model_vllm(cfg: EvalConfig):
    global llm_engine, tokenizer
    from vllm import LLM
    model_path = cfg.awq_model_id
    llm_engine = LLM(
        model=model_path,
        tokenizer=cfg.base_model_id,
        trust_remote_code=True,
        max_model_len=cfg.vllm_max_model_len,
        gpu_memory_utilization=cfg.vllm_gpu_memory_utilization,
        dtype='float16',
        quantization='awq',
    )
    tokenizer = llm_engine.get_tokenizer()
    print("vLLM Engine loaded.")


if config.inference_mode == 'hf':
    load_model_hf(config)
else:
    load_model_vllm(config)

In [ ]:
# ============================================================
# 4-B. Sanity Check: 1개 샘플로 모델 동작 검증
# ============================================================
import json

with open(config.test_data_path, 'r', encoding='utf-8') as f:
    sanity_item = json.loads(f.readline().strip())

messages = [
    {"role": "system", "content": config.system_message},
    {"role": "user", "content": f"{sanity_item['instruction']}\n\n{sanity_item['input'][:300]}"},
]

try:
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    print(f"Chat template 적용 성공")
    print(f"프롬프트 앞부분:\n{prompt[:300]}...\n")
except Exception as e:
    print(f"Chat template 실패: {e}")
    prompt = f"[|system|]{config.system_message}[|user|]{sanity_item['instruction']}\n\n{sanity_item['input'][:300]}[|assistant|]"
    print("Fallback 템플릿 사용")

# EOS 토큰 ID 확인 (EXAONE: eos_token + endofturn 토큰)
eos_ids = [tokenizer.eos_token_id]
for special_tok in ['[|endofturn|]', '<|endofturn|>']:
    tok_id = tokenizer.convert_tokens_to_ids(special_tok)
    if tok_id is not None and tok_id != tokenizer.unk_token_id and tok_id not in eos_ids:
        eos_ids.append(tok_id)
print(f"EOS token IDs: {eos_ids}")
print(f"  eos_token: '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")

if config.inference_mode == 'hf':
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    input_len = inputs.input_ids.shape[1]
    print(f"입력 토큰 수: {input_len}")

    import time
    try:
        start = time.perf_counter()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                repetition_penalty=config.repetition_penalty,
                eos_token_id=eos_ids,
            )
        elapsed = time.perf_counter() - start

        generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        gen_tokens = outputs[0].shape[0] - input_len
        print(f"생성 토큰 수: {gen_tokens}")
        print(f"소요 시간: {elapsed:.1f}s ({gen_tokens/elapsed:.1f} tok/s)")
        print(f"920 샘플 예상 시간: {elapsed * 920 / 3600:.1f}h")
        print(f"\n--- 생성 결과 (첫 500자) ---")
        print(generated[:500])
        print(f"\n--- 참조 답변 (첫 200자) ---")
        print(sanity_item['output'][:200])

        if gen_tokens >= 255:
            print("\n[경고] max_new_tokens까지 생성됨 - EOS 미인식 가능성")
        else:
            print(f"\n[OK] 정상 종료 (EOS 인식, {gen_tokens} tokens)")

    except Exception as e:
        print(f"\n[ERROR] model.generate() 실패!")
        print(f"  에러 타입: {type(e).__name__}")
        print(f"  에러 내용: {e}")
        import traceback
        traceback.print_exc()
else:
    print("vLLM 모드 - sanity check는 전체 평가에서 수행됩니다.")

In [ ]:
# (Sanity Check 통과 후 진행)

## 5. Generation Functions

In [ ]:
import time
import traceback
from tqdm.auto import tqdm

# EOS 토큰 ID 리스트 (sanity check에서 이미 설정됨)
def get_eos_ids():
    """EXAONE의 모든 EOS 토큰 ID를 반환"""
    ids = [tokenizer.eos_token_id]
    for tok in ['[|endofturn|]', '<|endofturn|>']:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != tokenizer.unk_token_id and tid not in ids:
            ids.append(tid)
    return ids

def build_prompt(item: Dict, cfg: EvalConfig) -> str:
    """Build prompt using EXAONE chat template."""
    if 'tokenizer' not in globals() or tokenizer is None:
        raise NameError('Tokenizer가 로드되지 않았습니다. 모델 로딩 셀을 먼저 실행하세요.')

    try:
        messages = [
            {"role": "system", "content": cfg.system_message},
            {"role": "user", "content": f"{item['instruction']}\n\n{item['input']}"},
        ]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except:
            return f"[|system|]{cfg.system_message}[|user|]{item['instruction']}\n\n{item['input']}[|assistant|]"
    except:
        return ""

def generate_hf_single(prompt: str, cfg: EvalConfig) -> tuple:
    if not prompt:
        return "", 0.0
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=cfg.vllm_max_model_len).to(model.device)
    input_len = inputs.input_ids.shape[1]
    start = time.perf_counter()
    with torch.no_grad():
        gen_kwargs = dict(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            repetition_penalty=cfg.repetition_penalty,
            do_sample=cfg.do_sample,
            eos_token_id=get_eos_ids(),
        )
        if cfg.do_sample and cfg.temperature > 0:
            gen_kwargs["temperature"] = cfg.temperature
            gen_kwargs["top_p"] = cfg.top_p
        outputs = model.generate(**gen_kwargs)
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True), time.perf_counter() - start

def generate_hf_batch(items: List[Dict], cfg: EvalConfig) -> tuple:
    gens, lats = [], []
    for item in tqdm(items, desc="HF generation"):
        try:
            t, l = generate_hf_single(build_prompt(item, cfg), cfg)
            gens.append(t); lats.append(l)
        except Exception as e:
            print(f"ERROR: {e}"); gens.append(""); lats.append(0.0)
    return gens, lats

def generate_vllm_batch(items: List[Dict], cfg: EvalConfig) -> tuple:
    from vllm import SamplingParams
    prompts = [p for p in [build_prompt(i, cfg) for i in items] if p]
    params = SamplingParams(
        temperature=cfg.temperature if cfg.do_sample else 0.0,
        top_p=cfg.top_p if cfg.do_sample else 1.0,
        max_tokens=cfg.max_new_tokens,
        repetition_penalty=cfg.repetition_penalty,
        stop=["[|user|]", "[|system|]", "[|assistant|]", "[|endofturn|]", "</s>", "<|endoftext|>"],
    )
    print(f"Running vLLM batch inference on {len(prompts)} prompts...")
    start = time.perf_counter()
    try:
        outputs = llm_engine.generate(prompts, params)
        total_time = time.perf_counter() - start
        return [o.outputs[0].text for o in outputs], [total_time/max(1, len(items))] * len(items)
    except Exception as e:
        print(f"vLLM ERROR: {e}"); return [""] * len(items), [0.0] * len(items)

def generate_batch(items: List[Dict], cfg: EvalConfig) -> tuple:
    if cfg.inference_mode == "hf": return generate_hf_batch(items, cfg)
    return generate_vllm_batch(items, cfg)

print("Generation functions ready.")

## 6. Metric Computation

In [ ]:
import sacrebleu
from rouge_score import rouge_scorer
import bert_score
import numpy as np


def compute_sacrebleu(hypotheses: List[str], references: List[str]) -> dict:
    """Compute corpus-level SacreBLEU with proper brevity penalty.
    Uses intl tokenization for better Korean handling."""
    bleu = sacrebleu.corpus_bleu(hypotheses, [references], tokenize='intl')
    return {
        "sacrebleu_score": bleu.score,
        "sacrebleu_bp": bleu.bp,
        "sacrebleu_precisions": bleu.precisions,
        "sacrebleu_signature": str(bleu),
    }


def compute_rouge_l(hypotheses: List[str], references: List[str]) -> dict:
    """Compute ROUGE-L F-measure per sample and corpus average."""
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    scores = []
    for ref, hyp in zip(references, hypotheses):
        result = scorer.score(ref, hyp)
        scores.append(result["rougeL"].fmeasure)
    return {
        "rouge_l_f1_mean": np.mean(scores) * 100,
        "rouge_l_f1_std": np.std(scores) * 100,
        "rouge_l_f1_per_sample": [s * 100 for s in scores],
    }


def compute_bertscore(hypotheses: List[str], references: List[str],
                      model_type: str = "bert-base-multilingual-cased") -> dict:
    """Compute BERTScore F1 using multilingual BERT."""
    P, R, F1 = bert_score.score(
        hypotheses, references,
        model_type=model_type,
        lang="ko",
        verbose=True,
    )
    return {
        "bertscore_f1_mean": F1.mean().item() * 100,
        "bertscore_f1_std": F1.std().item() * 100,
        "bertscore_precision_mean": P.mean().item() * 100,
        "bertscore_recall_mean": R.mean().item() * 100,
        "bertscore_f1_per_sample": (F1 * 100).tolist(),
    }


def compute_all_metrics(hypotheses: List[str], references: List[str],
                         latencies: List[float], cfg: EvalConfig) -> dict:
    """Compute all metrics with appropriate normalization per metric type."""

    # SacreBLEU: 경량 정규화 (문장부호 유지 - SacreBLEU 자체 토크나이저 활용)
    light_hyps = [normalize_light(h) or "(empty)" for h in hypotheses]
    light_refs = [normalize_light(r) or "(empty)" for r in references]

    # ROUGE-L, BERTScore: 풀 정규화
    norm_hyps = [normalize_text(h) or "(empty)" for h in hypotheses]
    norm_refs = [normalize_text(r) or "(empty)" for r in references]

    print("Computing SacreBLEU (light normalization, intl tokenizer)...")
    bleu_metrics = compute_sacrebleu(light_hyps, light_refs)

    print("Computing ROUGE-L (full normalization)...")
    rouge_metrics = compute_rouge_l(norm_hyps, norm_refs)

    print("Computing BERTScore (full normalization)...")
    bert_metrics = compute_bertscore(norm_hyps, norm_refs, model_type=cfg.bertscore_model)

    # Latency metrics
    valid_latencies = [l for l in latencies if l > 0]
    latency_metrics = {
        "avg_latency_sec": np.mean(valid_latencies) if valid_latencies else 0,
        "p50_latency_sec": np.percentile(valid_latencies, 50) if valid_latencies else 0,
        "p95_latency_sec": np.percentile(valid_latencies, 95) if valid_latencies else 0,
    }

    # Generation length stats (on light-normalized text for accurate comparison)
    gen_lengths = [len(h) for h in light_hyps]
    ref_lengths = [len(r) for r in light_refs]
    length_metrics = {
        "avg_gen_length_chars": np.mean(gen_lengths),
        "avg_ref_length_chars": np.mean(ref_lengths),
        "length_ratio": np.mean(gen_lengths) / max(np.mean(ref_lengths), 1),
    }

    return {
        **bleu_metrics,
        **rouge_metrics,
        **bert_metrics,
        **latency_metrics,
        **length_metrics,
    }


print("Metric computation functions defined.")

## 7. Run Full Evaluation

In [ ]:
# ============================================================
# 7-A. Generate predictions
# ============================================================
print(f"Generating predictions for {len(test_data)} samples...")
print(f"  max_new_tokens     = {config.max_new_tokens}")
print(f"  temperature        = {config.temperature}")
print(f"  top_p              = {config.top_p}")
print(f"  repetition_penalty = {config.repetition_penalty}")
print(f"  do_sample          = {config.do_sample}")
print()

generations, latencies = generate_batch(test_data, config)

references = [item["output"] for item in test_data]

print(f"\nGeneration complete. {len(generations)} outputs produced.")
print(f"Total time: {sum(latencies):.1f}s")

In [ ]:
# ============================================================
# 7-B. Compute all metrics (corpus-level)
# ============================================================
corpus_metrics = compute_all_metrics(generations, references, latencies, config)

print("\n" + "=" * 60)
print("CORPUS-LEVEL RESULTS")
print("=" * 60)
print(f"  SacreBLEU          : {corpus_metrics['sacrebleu_score']:.2f}")
print(f"  BLEU Brevity Pen.  : {corpus_metrics['sacrebleu_bp']:.4f}")
print(f"  BLEU Precisions    : {[f'{p:.1f}' for p in corpus_metrics['sacrebleu_precisions']]}")
print(f"  ROUGE-L F1         : {corpus_metrics['rouge_l_f1_mean']:.2f} (+/- {corpus_metrics['rouge_l_f1_std']:.2f})")
print(f"  BERTScore F1       : {corpus_metrics['bertscore_f1_mean']:.2f} (+/- {corpus_metrics['bertscore_f1_std']:.2f})")
print(f"  BERTScore P / R    : {corpus_metrics['bertscore_precision_mean']:.2f} / {corpus_metrics['bertscore_recall_mean']:.2f}")
print(f"  Avg Latency        : {corpus_metrics['avg_latency_sec']:.3f}s")
print(f"  p50 Latency        : {corpus_metrics['p50_latency_sec']:.3f}s")
print(f"  p95 Latency        : {corpus_metrics['p95_latency_sec']:.3f}s")
print(f"  Avg Gen Length     : {corpus_metrics['avg_gen_length_chars']:.0f} chars")
print(f"  Avg Ref Length     : {corpus_metrics['avg_ref_length_chars']:.0f} chars")
print(f"  Length Ratio       : {corpus_metrics['length_ratio']:.2f}")
print(f"  SacreBLEU sig      : {corpus_metrics['sacrebleu_signature']}")
print("=" * 60)

## 8. Category-Level Evaluation

In [ ]:
from collections import defaultdict


def compute_category_metrics(test_data: List[Dict],
                              generations: List[str],
                              references: List[str],
                              latencies: List[float],
                              cfg: EvalConfig) -> pd.DataFrame:
    """Compute metrics grouped by category."""
    # Group indices by category
    cat_indices = defaultdict(list)
    for i, item in enumerate(test_data):
        cat_indices[item["category"]].append(i)

    rows = []
    for cat, indices in sorted(cat_indices.items()):
        cat_gens = [generations[i] for i in indices]
        cat_refs = [references[i] for i in indices]
        cat_lats = [latencies[i] for i in indices]

        # Normalize
        norm_hyps = [normalize_text(h) or "(empty)" for h in cat_gens]
        norm_refs = [normalize_text(r) or "(empty)" for r in cat_refs]

        # SacreBLEU
        bleu = sacrebleu.corpus_bleu(norm_hyps, [norm_refs])

        # ROUGE-L
        r_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
        rouge_scores = [r_scorer.score(r, h)["rougeL"].fmeasure for r, h in zip(norm_refs, norm_hyps)]

        # BERTScore (skip if category too small to be meaningful)
        if len(norm_hyps) >= 2:
            P, R, F1 = bert_score.score(
                norm_hyps, norm_refs,
                model_type=cfg.bertscore_model,
                lang="ko", verbose=False,
            )
            bs_f1 = F1.mean().item() * 100
        else:
            bs_f1 = float('nan')

        rows.append({
            "category": cat,
            "n_samples": len(indices),
            "sacrebleu": bleu.score,
            "rouge_l_f1": np.mean(rouge_scores) * 100,
            "bertscore_f1": bs_f1,
            "avg_latency": np.mean(cat_lats),
            "avg_gen_len": np.mean([len(h) for h in norm_hyps]),
            "avg_ref_len": np.mean([len(r) for r in norm_refs]),
        })

    df = pd.DataFrame(rows)
    return df


cat_df = compute_category_metrics(test_data, generations, references, latencies, config)

print("\n" + "=" * 90)
print("CATEGORY-LEVEL RESULTS")
print("=" * 90)
print(cat_df.to_string(index=False, float_format="{:.2f}".format))

# Identify strongest / weakest
if len(cat_df) > 1:
    best_cat = cat_df.loc[cat_df["sacrebleu"].idxmax()]
    worst_cat = cat_df.loc[cat_df["sacrebleu"].idxmin()]
    print(f"\nStrongest category (BLEU): {best_cat['category']} ({best_cat['sacrebleu']:.2f})")
    print(f"Weakest  category (BLEU): {worst_cat['category']} ({worst_cat['sacrebleu']:.2f})")
else:
    print("\nOnly one category found - no comparison possible.")
    print("NOTE: If all samples are 'other', see Issue #70 for test set quality improvement.")

## 9. Decoding Parameter Grid Search

In [ ]:
# ============================================================
# 9-A. Grid search configuration
# ============================================================
import itertools
from dataclasses import replace


param_grid = {
    "max_new_tokens": [128, 150, 200],
    "temperature": [0.3, 0.5],
    "repetition_penalty": [1.1, 1.3, 1.5],
    "top_p": [0.85, 0.95],
}

# Total combinations
n_combos = 1
for v in param_grid.values():
    n_combos *= len(v)
print(f"Total parameter combinations: {n_combos}")
print(f"Samples per combination: {config.grid_search_n_samples}")
print(f"Grid search enabled: {config.grid_search_enabled}")
if not config.grid_search_enabled:
    print("\nSet config.grid_search_enabled = True to run this section.")

In [ ]:
# ============================================================
# 9-B. Run grid search
# ============================================================

grid_results = []

if config.grid_search_enabled:
    # Subset of test data for grid search
    grid_subset = test_data[:config.grid_search_n_samples]
    grid_refs = [item["output"] for item in grid_subset]

    keys = list(param_grid.keys())
    combos = list(itertools.product(*[param_grid[k] for k in keys]))

    for combo_idx, combo in enumerate(combos):
        params = dict(zip(keys, combo))
        print(f"\n--- Combo {combo_idx + 1}/{len(combos)}: {params} ---")

        # Create a temporary config with these params
        grid_cfg = replace(config, **params)

        try:
            gens, lats = generate_batch(grid_subset, grid_cfg)

            # Quick metrics (BLEU + ROUGE-L only for speed)
            norm_hyps = [normalize_text(h) or "(empty)" for h in gens]
            norm_refs = [normalize_text(r) or "(empty)" for r in grid_refs]

            bleu = sacrebleu.corpus_bleu(norm_hyps, [norm_refs])
            r_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
            rouge_scores = [r_scorer.score(r, h)["rougeL"].fmeasure for r, h in zip(norm_refs, norm_hyps)]

            result = {
                **params,
                "sacrebleu": bleu.score,
                "brevity_penalty": bleu.bp,
                "rouge_l_f1": np.mean(rouge_scores) * 100,
                "avg_latency": np.mean(lats),
                "avg_gen_len": np.mean([len(h) for h in norm_hyps]),
            }
            grid_results.append(result)
            print(f"  BLEU={result['sacrebleu']:.2f}, ROUGE-L={result['rouge_l_f1']:.2f}, "
                  f"BP={result['brevity_penalty']:.3f}, Latency={result['avg_latency']:.2f}s")

        except Exception as e:
            print(f"  FAILED: {e}")
            grid_results.append({**params, "sacrebleu": 0, "rouge_l_f1": 0, "error": str(e)})

    # Summary
    grid_df = pd.DataFrame(grid_results)
    grid_df = grid_df.sort_values("sacrebleu", ascending=False)
    print("\n" + "=" * 90)
    print("GRID SEARCH RESULTS (sorted by BLEU)")
    print("=" * 90)
    print(grid_df.head(10).to_string(index=False, float_format="{:.2f}".format))

    best = grid_df.iloc[0]
    print(f"\nBest configuration:")
    for k in keys:
        print(f"  {k}: {best[k]}")
    print(f"  -> BLEU={best['sacrebleu']:.2f}, ROUGE-L={best['rouge_l_f1']:.2f}")
else:
    print("Grid search skipped. Set config.grid_search_enabled = True to run.")

## 10. W&B Logging & Visualization

In [ ]:
# ============================================================
# 10-A. Initialize W&B run
# ============================================================
from datetime import datetime

run_name = config.wandb_run_name or f"eval-{config.inference_mode}-{datetime.now().strftime('%m%d-%H%M')}"

if config.inference_mode == "vllm":
    evaluated_model = config.awq_model_id
else:
    evaluated_model = f"{config.base_model_id} + {config.lora_adapter_id}"

run = wandb.init(
    project=config.wandb_project,
    entity=config.wandb_entity,
    name=run_name,
    tags=config.wandb_tags,
    config={
        "inference_mode": config.inference_mode,
        "model": evaluated_model,
        "base_model": config.base_model_id,
        "lora_adapter": config.lora_adapter_id,
        "max_new_tokens": config.max_new_tokens,
        "temperature": config.temperature,
        "top_p": config.top_p,
        "repetition_penalty": config.repetition_penalty,
        "do_sample": config.do_sample,
        "system_message": config.system_message,
        "n_test_samples": len(test_data),
        "bertscore_model": config.bertscore_model,
        "normalization": "thought_tags + PII_tokens + whitespace",
    },
)

print(f"W&B run: {run.name} ({run.url})")

In [ ]:
# ============================================================
# 10-B. Log corpus-level metrics
# ============================================================

# Remove per-sample lists from the metrics dict before logging scalars
scalar_metrics = {
    k: v for k, v in corpus_metrics.items()
    if not isinstance(v, (list, str))
}
wandb.log(scalar_metrics)

# Also log the full signature as a summary
wandb.run.summary["sacrebleu_signature"] = corpus_metrics.get("sacrebleu_signature", "")

print("Corpus-level metrics logged to W&B.")

In [ ]:
# ============================================================
# 10-C. Log per-sample results as W&B Table
# ============================================================

columns = [
    "id", "category",
    "reference", "prediction",
    "ref_normalized", "pred_normalized",
    "rouge_l_f1", "bertscore_f1",
    "ref_length", "pred_length", "latency_sec",
]

table_data = []
rouge_per_sample = corpus_metrics.get("rouge_l_f1_per_sample", [0] * len(test_data))
bert_per_sample = corpus_metrics.get("bertscore_f1_per_sample", [0] * len(test_data))

for i, item in enumerate(test_data):
    norm_ref = normalize_text(references[i])
    norm_pred = normalize_text(generations[i])
    table_data.append([
        item.get("id", f"sample_{i}"),
        item.get("category", "unknown"),
        references[i][:500],
        generations[i][:500],
        norm_ref[:500],
        norm_pred[:500],
        rouge_per_sample[i] if i < len(rouge_per_sample) else 0,
        bert_per_sample[i] if i < len(bert_per_sample) else 0,
        len(norm_ref),
        len(norm_pred),
        latencies[i],
    ])

results_table = wandb.Table(columns=columns, data=table_data)
wandb.log({"per_sample_results": results_table})

print(f"Per-sample results table logged ({len(table_data)} rows).")

In [ ]:
# ============================================================
# 10-D. Log category-level breakdown
# ============================================================

cat_table = wandb.Table(dataframe=cat_df)
wandb.log({"category_metrics": cat_table})

# Log category metrics as individual scalars for easy comparison
for _, row in cat_df.iterrows():
    cat_name = row["category"]
    wandb.run.summary[f"cat/{cat_name}/sacrebleu"] = row["sacrebleu"]
    wandb.run.summary[f"cat/{cat_name}/rouge_l_f1"] = row["rouge_l_f1"]
    wandb.run.summary[f"cat/{cat_name}/bertscore_f1"] = row["bertscore_f1"]
    wandb.run.summary[f"cat/{cat_name}/n_samples"] = row["n_samples"]

print("Category-level metrics logged to W&B.")

In [ ]:
# ============================================================
# 10-E. Log grid search results (if available)
# ============================================================

if grid_results:
    grid_table = wandb.Table(dataframe=pd.DataFrame(grid_results))
    wandb.log({"grid_search_results": grid_table})

    # Log best config
    best_grid = max(grid_results, key=lambda x: x.get("sacrebleu", 0))
    for k, v in best_grid.items():
        if k not in ("error",):
            wandb.run.summary[f"best_grid/{k}"] = v
    print("Grid search results logged to W&B.")
else:
    print("No grid search results to log.")

In [ ]:
# ============================================================
# 10-F. Finish W&B run
# ============================================================
wandb.finish()
print("W&B run finished.")

## 11. Results Report & Export

In [ ]:
# ============================================================
# 11-A. Save results JSON
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
result_filename = f"eval_{config.inference_mode}_{timestamp}.json"
result_path = os.path.join(config.output_dir, result_filename)

# Build export dict (remove non-serializable items)
export_data = {
    "metadata": {
        "timestamp": timestamp,
        "inference_mode": config.inference_mode,
        "model": evaluated_model,
        "base_model": config.base_model_id,
        "n_test_samples": len(test_data),
        "system_message": config.system_message,
        "normalization": "thought_tags + PII_tokens + whitespace",
    },
    "generation_config": {
        "max_new_tokens": config.max_new_tokens,
        "temperature": config.temperature,
        "top_p": config.top_p,
        "repetition_penalty": config.repetition_penalty,
        "do_sample": config.do_sample,
    },
    "corpus_metrics": {
        k: v for k, v in corpus_metrics.items()
        if not isinstance(v, list)
    },
    "category_metrics": cat_df.to_dict(orient="records"),
}

if grid_results:
    export_data["grid_search"] = {
        "param_grid": param_grid,
        "n_samples_per_combo": config.grid_search_n_samples,
        "results": grid_results,
    }

with open(result_path, "w", encoding="utf-8") as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2, default=str)

print(f"Results saved to: {result_path}")

In [ ]:
# ============================================================
# 11-A-2. Auto-download results JSON (Colab 런타임 종료 대비)
# ============================================================

try:
    from google.colab import files
    files.download(result_path)
    print(f"다운로드 시작: {result_path}")
except ImportError:
    print("Colab 환경이 아닙니다. 수동으로 파일을 복사하세요.")

In [ ]:
# ============================================================
# 11-B. Print formatted summary
# ============================================================

# M3 KPI targets for comparison
kpi_targets = {
    "sacrebleu_score": {"target": 30, "label": "SacreBLEU"},
    "rouge_l_f1_mean": {"target": 40, "label": "ROUGE-L F1"},
    "bertscore_f1_mean": {"target": 55, "label": "BERTScore F1"},
    "p50_latency_sec": {"target": 2.0, "label": "p50 Latency (s)", "lower_is_better": True},
}

print("\n" + "=" * 70)
print("EVALUATION SUMMARY vs M3 KPI TARGETS")
print("=" * 70)
print(f"{'Metric':<25} {'Result':>10} {'Target':>10} {'Status':>10}")
print("-" * 70)

for metric_key, info in kpi_targets.items():
    value = corpus_metrics.get(metric_key, 0)
    target = info["target"]
    lower_better = info.get("lower_is_better", False)

    if lower_better:
        met = value <= target
    else:
        met = value >= target

    status = "PASS" if met else "FAIL"
    print(f"  {info['label']:<23} {value:>10.2f} {target:>10.1f} {status:>10}")

print("=" * 70)

In [ ]:
# ============================================================
# 11-C. Show sample predictions vs references
# ============================================================

n_show = min(5, len(test_data))
print(f"\nSample predictions (showing {n_show} examples):")
print("=" * 70)

for i in range(n_show):
    item = test_data[i]
    norm_ref = normalize_text(references[i])
    norm_gen = normalize_text(generations[i])

    rouge_score_i = corpus_metrics["rouge_l_f1_per_sample"][i] if i < len(corpus_metrics.get("rouge_l_f1_per_sample", [])) else 0
    bert_score_i = corpus_metrics["bertscore_f1_per_sample"][i] if i < len(corpus_metrics.get("bertscore_f1_per_sample", [])) else 0

    print(f"\n--- [{i}] ID: {item.get('id', '?')} | Category: {item.get('category', '?')} ---")
    print(f"  ROUGE-L: {rouge_score_i:.1f} | BERTScore: {bert_score_i:.1f}")
    print(f"  REF ({len(norm_ref)} chars): {norm_ref[:200]}{'...' if len(norm_ref) > 200 else ''}")
    print(f"  GEN ({len(norm_gen)} chars): {norm_gen[:200]}{'...' if len(norm_gen) > 200 else ''}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# 11-D. Recommendations
# ============================================================

print("\nRECOMMENDATIONS")
print("=" * 70)

recommendations = []

# Check BLEU
bleu_val = corpus_metrics.get("sacrebleu_score", 0)
bp_val = corpus_metrics.get("sacrebleu_bp", 1.0)
if bleu_val < 30:
    if bp_val < 0.8:
        recommendations.append(
            f"BLEU ({bleu_val:.1f}) below target (30). Brevity penalty is {bp_val:.3f} - "
            "generations may be too short. Try increasing max_new_tokens or reducing "
            "repetition_penalty."
        )
    else:
        recommendations.append(
            f"BLEU ({bleu_val:.1f}) below target (30). Brevity penalty is fine ({bp_val:.3f}). "
            "Focus on data quality: high PII density and category imbalance are likely root causes. "
            "See Issue #70."
        )

# Check ROUGE-L
rouge_val = corpus_metrics.get("rouge_l_f1_mean", 0)
if rouge_val < 40:
    recommendations.append(
        f"ROUGE-L ({rouge_val:.1f}) below target (40). Investigate length ratio: "
        f"{corpus_metrics.get('length_ratio', 0):.2f}. If >> 1.0, model over-generates. "
        "Tune max_new_tokens closer to avg reference length."
    )

# Check BERTScore
bert_val = corpus_metrics.get("bertscore_f1_mean", 0)
if bert_val < 55:
    recommendations.append(
        f"BERTScore F1 ({bert_val:.1f}) below target (55). This indicates semantic mismatch. "
        "Consider RAG augmentation (Issue #54) to provide relevant context during generation."
    )

# Check latency
p50_lat = corpus_metrics.get("p50_latency_sec", 0)
if p50_lat > 2.0:
    recommendations.append(
        f"p50 latency ({p50_lat:.2f}s) above target (2.0s). Use AWQ+vLLM mode, "
        "or reduce max_new_tokens. See Issue #69."
    )

# Check category diversity
n_categories = len(cat_df)
if n_categories <= 1:
    recommendations.append(
        "Only 1 category found in test data. Category-level analysis is not meaningful. "
        "Rebuild test set with diverse categories (Issue #70)."
    )

# Length ratio
length_ratio = corpus_metrics.get("length_ratio", 1.0)
if length_ratio > 2.0:
    recommendations.append(
        f"Length ratio is {length_ratio:.1f}x (gen/ref). Model over-generates significantly. "
        f"Set max_new_tokens closer to avg ref length ({corpus_metrics.get('avg_ref_length_chars', 0):.0f} chars)."
    )

if not recommendations:
    print("All KPI targets met. No further recommendations.")
else:
    for i, rec in enumerate(recommendations, 1):
        print(f"  {i}. {rec}")

print("\n" + "=" * 70)
print(f"Results file: {result_path}")
print("Pipeline complete.")